# 用两套房，找出同一套价格公式

试用模型 `价格 = alpha * 面积 + beta * 距离`。价格用万元，面积用平方米，到重点小学的距离用公里。甲房为 `(80,1)`，价格 150；乙房为 `(100,2)`，价格 180。数据是虚构的，模型暂不含常数项，也不考虑其他房屋条件。

面积、距离和价格已知，待求的是两个模型系数。这次 `A` 装房屋数据，`x=(alpha,beta)` 装系数，`b` 装已知价格。依赖为 NumPy、Matplotlib；ipywidgets 提供滑块，也可以直接改函数参数。请从第一格开始顺序运行。

先猜一组系数 `(1.8,-5)`。它们会给甲、乙两房算出什么价格？先手算，再运行。

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

A = np.array([[80., 1.], [100., 2.]])
b = np.array([150., 180.])
trial = np.array([1.8, -5.])
print('房屋数据 A =\n', A)
print('模型系数 x =', trial)
print('模型价格 Ax =', A @ trial)
print('已知价格 b  =', b)
print('差 Ax - b   =', A @ trial - b)

## 先把甲房算对

甲房只要求 `80*alpha + beta = 150`，所以 `beta = 150 - 80*alpha`。把 alpha 分别取 1.5、2、2.5，先算出各自的 beta。每组都能算对甲房，它们给乙房的价格会一样吗？

In [ ]:
for alpha in [1.5, 2., 2.5]:
    beta = 150 - 80 * alpha
    candidate = np.array([alpha, beta])
    print('系数', candidate, '-> 甲、乙房价格', A @ candidate)

## 两条约束的交集

每套房给出关于 alpha、beta 的一条方程。把第一条乘 2，再减去第二条，得到 `60*alpha=120`。先在纸上求 alpha，再代回 `80*alpha+beta=150` 求 beta。

下面只执行刚才的消元算式，再代回检查。矩阵乘法负责核对，不替我们选择消元步骤。

In [ ]:
alpha = (2 * b[0] - b[1]) / (2 * A[0, 0] - A[1, 0])
beta = b[0] - A[0, 0] * alpha
solution = np.array([alpha, beta])
print('消元得到的系数 x =', solution)
print('两套房的模型价格 =', A @ solution)
print('是否都算对？     =', np.allclose(A @ solution, b))

## 移动系数，观察两个价格误差

左图的横轴是面积系数 alpha，纵轴是距离系数 beta。图上的一个点代表一整套价格公式，两个坐标并不是房屋的面积和距离。两条直线分别收集能算对甲房、乙房的所有系数组合。

右图比较已知价格和当前模型价格。先让圆点停在第一条线上，再沿它移动：甲房始终算对时，乙房的价格怎样变化？试着让圆点落到交点，使两项误差同时为零。

In [ ]:
def inspect_rates(alpha=1.8, beta=-5.):
    candidate = np.array([alpha, beta])
    predicted = A @ candidate
    line_alpha = np.linspace(1.5, 2.5, 300)
    fig, (ax_rates, ax_prices) = plt.subplots(1, 2, figsize=(11, 4.5),
                                            dpi=160, layout='constrained')
    ax_rates.plot(line_alpha, 150 - 80 * line_alpha, color='#0072B2',
                  linewidth=2, label=r'House 1: $80\alpha+\beta=150$')
    ax_rates.plot(line_alpha, (180 - 100 * line_alpha) / 2, color='#D55E00',
                  linewidth=2, label=r'House 2: $100\alpha+2\beta=180$')
    ax_rates.scatter(*solution, color='#19856F', marker='*', s=170, zorder=4,
                     label='shared solution (2, -10)')
    ax_rates.scatter(*candidate, color='#B83F4E', s=60, edgecolor='white',
                     zorder=5, label='current coefficients')
    ax_rates.set(xlim=(1.5, 2.5), ylim=(-50, 30), title='Each house constrains the coefficients',
                 xlabel=r'$\alpha$ (10,000 CNY / square metre)',
                 ylabel=r'$\beta$ (10,000 CNY / kilometre)')
    ax_rates.grid(alpha=0.2)
    ax_rates.legend(fontsize=8, loc='upper right')
    positions = np.arange(2)
    ax_prices.bar(positions - 0.18, b, width=0.36, color='#647887', label='observed price')
    ax_prices.bar(positions + 0.18, predicted, width=0.36, color='#19856F', label='model price')
    ax_prices.set(xticks=positions, xticklabels=['House 1', 'House 2'], ylim=(0, 300),
                  ylabel='Price (10,000 CNY)', title='One formula must match both prices')
    ax_prices.grid(axis='y', alpha=0.2)
    ax_prices.set_axisbelow(True)
    ax_prices.legend(fontsize=9, loc='upper left')
    plt.show()
    print('系数 x       =', candidate)
    print('模型价格 Ax  =', predicted)
    print('价格误差 Ax-b=', predicted - b)

inspect_rates(1.8, -5)

In [ ]:
try:
    import ipywidgets as widgets
    from IPython.display import display
except ImportError:
    print('滑块需要 ipywidgets；也可修改 inspect_rates(1.8, -5) 中的参数后重新运行。')
else:
    display(widgets.interactive(
        inspect_rates,
        alpha=widgets.FloatSlider(value=1.8, min=1.5, max=2.5, step=0.01, description='alpha'),
        beta=widgets.FloatSlider(value=-5, min=-30, max=10, step=0.5, description='beta')))

## 系数确定后，算另一套房

丙房面积 90、距离 0.5，价格还不知道。保留刚求出的系数 `(2,-10)`，先手算价格，再运行。然后一次只改一个数：面积增加 10 平方米，模型价格增加多少？距离增加 1 公里，又改变多少？

这里 x 仍表示系数。房屋变了，A 换成丙房的数据；已知 A 和 x，计算得到新的 b。这个结果是简化模型的预测，实际成交价需要另外观察。

In [ ]:
area = 90.
distance = 0.5
A_new = np.array([[area, distance]])
print('丙房数据 A_new =', A_new)
print('已知系数 x     =', solution)
print('模型价格 b_new =', A_new @ solution, '万元')

## 自己说清楚这两个方向

给定甲、乙两房的面积、距离和价格，求的是哪两个量？给定丙房的数据和已有系数，又求的是什么？每一步的 A、x、b 各装哪些数？

再想一个模型的边界：如果两套房面积和距离完全相同，已知价格却不同，为什么这组方程无解？把两条方程相减，说明矛盾在哪里。